# 10 · 사진으로 음식 알아보기 — 펼친 신경망(Dense) vs 합성곱 신경망(CNN)
교과서 **부록 229~237쪽**(흰쌀밥과 흑미밥 분류 → 영양 정보 알려 주기)을 우리 수업 방식으로 옮겼습니다.

맨 먼저 **런타임 → 런타임 유형 변경 → T4 GPU** 를 고르세요(학습이 몇 배 빨라집니다). 그리고 **파일 → 드라이브에 사본 저장**.

사진 데이터는 두 가지 중 하나로 씁니다.
- **A. 바로 실행** — 공개 꽃 사진(데이지 · 해바라기) 두 종류를 자동으로 내려받아 같은 과정을 연습합니다.
- **B. 교과서 음식 사진** — 선생님이 공유한 드라이브 폴더 `food`(train/test 안에 01 흰쌀밥 · 07 흑미밥)를 내 드라이브에 바로가기로 추가한 뒤 B 셀을 실행합니다.

## ① 사진 준비 — A 또는 B 중 하나만 실행

In [ ]:
# A. 바로 실행 — 꽃 사진 두 종류
import tensorflow as tf, pathlib, shutil, random, os
p = pathlib.Path(tf.keras.utils.get_file('flower_photos', 'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz', untar=True))
# 텐서플로 판마다 풀리는 위치가 달라서 «daisy» 폴더가 있는 곳을 찾습니다
src = next(d.parent for d in list(p.parent.rglob('daisy')) if d.is_dir())
random.seed(0)
for cls in ['daisy', 'sunflowers']:
    files = sorted((src / cls).glob('*.jpg')); random.shuffle(files)
    for part, chunk in (('train', files[:200]), ('test', files[200:260])):
        d = pathlib.Path('/content/food') / part / cls; d.mkdir(parents=True, exist_ok=True)
        for f in chunk: shutil.copy(f, d)
train_dir, test_dir = '/content/food/train', '/content/food/test'
print(os.listdir(train_dir))

In [ ]:
# B. 교과서 음식 사진 — 선생님 공유 폴더를 «드라이브에 바로가기 추가» 한 뒤
from google.colab import drive
drive.mount('/content/drive')
train_dir = '/content/drive/MyDrive/food/train'     # 폴더 위치가 다르면 고치세요
test_dir  = '/content/drive/MyDrive/food/test'

## ② 사진을 숫자로 — 크기 맞추고 0~1로 나누기
💡 픽셀 값 0~255 를 0~1 로 → `1/255`. 사진 크기는 150×150.

In [ ]:
import tensorflow as tf
size = (150, 150)
train = tf.keras.utils.image_dataset_from_directory(train_dir, image_size=size, batch_size=32, label_mode='categorical', seed=1)
test  = tf.keras.utils.image_dataset_from_directory(test_dir,  image_size=size, batch_size=32, label_mode='categorical', shuffle=False)
names = train.class_names
print(names)
scale = tf.keras.layers.Rescaling(____)
train = train.map(lambda x, y: (scale(x), y)); test = test.map(lambda x, y: (scale(x), y))

## ③ 모델 1 — 사진을 한 줄로 펼쳐 넣는 신경망(교과서 233쪽)
150×150×3 = **67,500개** 숫자를 한 줄로 펼쳐(Flatten) 은닉층 두 개에 넣습니다.
💡 출력층 뉴런 수 = 분류할 종류 수 → `len(names)`. 여러 종류 중 하나 → `softmax`.

In [ ]:
from tensorflow.keras import layers, models
m1 = models.Sequential([
    layers.Input(shape=(150, 150, 3)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(____, activation='softmax'),
])
m1.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
m1.summary()
h1 = m1.fit(train, epochs=10, validation_data=test)

## ④ 모델 2 — 합성곱 신경망 CNN(교과서 236쪽)
사진을 펼치기 전에 **작은 창(필터)으로 훑어 무늬를 찾고(Conv2D) → 줄이는(MaxPool)** 층을 넣습니다.
💡 필터 크기는 3×3 → `kernel_size=3`.

In [ ]:
m2 = models.Sequential([
    layers.Input(shape=(150, 150, 3)),
    layers.Conv2D(32, kernel_size=____, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Conv2D(64, kernel_size=3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(len(names), activation='softmax'),
])
m2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
m2.summary()
h2 = m2.fit(train, epochs=10, validation_data=test)

## ⑤ 비교 — 학습 곡선과 테스트 정확도

In [ ]:
import matplotlib.pyplot as plt
for h, name in ((h1, 'Dense'), (h2, 'CNN')):
    plt.plot(h.history['val_accuracy'], label=name)
plt.xlabel('epoch'); plt.ylabel('test accuracy'); plt.legend(); plt.show()
print('Dense :', m1.evaluate(test, verbose=0)[1])
print('CNN   :', m2.evaluate(test, verbose=0)[1])
print('가중치 수 — Dense', m1.count_params(), '/ CNN', m2.count_params())

**생각해 보기** 어느 모델이 더 정확한가요? 가중치(파라미터) 수는 어느 쪽이 더 많나요? 왜 사진에는 CNN이 유리할까요?

## ⑥ (선택) 내 사진 한 장 분류하기

In [ ]:
from google.colab import files
import numpy as np
up = files.upload(); fname = list(up.keys())[0]
img = tf.keras.utils.load_img(fname, target_size=(150, 150))
x = tf.keras.utils.img_to_array(img)[None] / 255
p = m2.predict(x)[0]
print({n: round(float(v), 3) for n, v in zip(names, p)}, '→', names[int(np.argmax(p))])

---
### 정답 예시 (막혔을 때만)
- ② `Rescaling(1/255)` · ③ `Dense(len(names), …)` · ④ `kernel_size=3`
- 보통 **CNN이 더 정확하면서 가중치는 훨씬 적습니다**. Dense는 픽셀 하나하나를 따로 보지만, CNN은 «가장자리 · 무늬»처럼 위치가 달라도 같은 특징을 찾기 때문입니다.
- 교과서 음식 사진은 종류마다 30장뿐이라 정확도가 들쭉날쭉합니다 — **데이터 양**이 성능을 좌우한다는 것을 확인할 수 있습니다.